In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model
from langchain_core import tools
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode,tools_condition



ImportError: cannot import name 'tool' from 'langchain_core' (c:\Users\Taimour\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_core\__init__.py)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
class State(TypedDict):
    messages:Annotated[list,add_messages]

In [ ]:
@tools
def get_stock_price(symbol->str)->float:
    return{
        "MSFT":200.3,
        "AMZN":150.0,
        "RIL":87.6
    }.get(symbol,0.0)

tools=[get_stock_price,0.0]



llm=init_chat_model("google_genai:gemini-3.6-flash")
llm_with_tools=llm.bind_tools(tools)



In [ ]:
def chatbot(state:State)->State:
    return {"messages":[llm.invoke(state["messages"])]}
builder=StateGraph(State)
builder.add_node("chatbot_node",chatbot)
builder.add_edge(START,"chatbot_node")
builder.add_conditional_edges("chatbot",tools_condition)
builder.add_edge("chatbot_node",END)
graph=builder.compile()

from IPython.display import Image,display
display(Image(graph.get_graph().draw_mermaid.png()))